# 83 — Historical Site-Year Fusion

Fuses emissions warehouse and documentary noncompliance into one site-year table.

In [ ]:
import os, io, re, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

phase80_cfg = load_yaml(CONFIGS / "phase80.yml")
permit_cfg = load_yaml(CONFIGS / "permit_registry.yml")
run_cfg = load_yaml(CONFIGS / "run.yml")

In [ ]:
step = "83_historical_site_year_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

wh, _ = safe_read_csv(OUTPUTS / "81_historical_emissions_warehouse" / "historical_emissions_warehouse.csv")
nc, _ = safe_read_csv(OUTPUTS / "82_documentary_noncompliance_warehouse" / "documentary_noncompliance_warehouse.csv")

if wh.empty:
    raise FileNotFoundError("Need outputs/81_historical_emissions_warehouse/historical_emissions_warehouse.csv")

exceedance_like = wh[wh["metric_std"].astype(str).str.contains("exceedance_count", case=False, na=False)].copy()
elv = wh[wh["metric_std"].astype(str) == "ELV"].copy()
means = wh[wh["metric_std"].astype(str).isin(["daily_mean","half_hourly_mean","p95","max"])].copy()

ex_perm = exceedance_like.groupby(["permit_id","facility_name","site_id","region_folder","report_year"], as_index=False)["value_max"].sum().rename(columns={"value_max":"exceedance_total"})
ratio = means.merge(elv[["permit_id","pollutant_std","report_year","value_mean"]].rename(columns={"value_mean":"elv_value"}), on=["permit_id","pollutant_std","report_year"], how="left")
ratio["ratio"] = ratio["value_max"] / ratio["elv_value"]
ratio_sum = ratio.groupby(["permit_id","facility_name","site_id","region_folder","report_year"], as_index=False)["ratio"].max().rename(columns={"ratio":"worst_ratio"})

base = wh.groupby(["permit_id","facility_name","site_id","region_folder","report_year"], as_index=False).agg(
    pollutant_count=("pollutant_std","nunique"),
    metric_rows=("metric_std","size")
)
base = base.merge(ex_perm, on=["permit_id","facility_name","site_id","region_folder","report_year"], how="left")
base = base.merge(ratio_sum, on=["permit_id","facility_name","site_id","region_folder","report_year"], how="left")

if not nc.empty:
    nc2 = nc[["permit_id","report_year","cems_hit_count","noncompliance_hits","incident_hits"]].copy()
    base = base.merge(nc2, on=["permit_id","report_year"], how="left")

for c in ["exceedance_total","worst_ratio","cems_hit_count","noncompliance_hits","incident_hits"]:
    if c not in base.columns:
        base[c] = 0
    base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

base["historical_risk_score"] = (
    base["exceedance_total"] * 3
    + base["worst_ratio"] * 10
    + base["noncompliance_hits"] * 0.3
    + base["incident_hits"] * 0.2
    + base["cems_hit_count"] * 0.01
)

base.to_csv(out / "historical_site_year_fusion.csv", index=False)
write_json(out / "historical_site_year_fusion_summary.json", {
    "rows": int(len(base)),
    "permits": int(base["permit_id"].nunique()),
    "years": int(base["report_year"].nunique()),
})

manifest = manifest_base(step, [CONFIGS / "permit_registry.yml"])
add_artifact(manifest, out / "historical_site_year_fusion.csv")
add_artifact(manifest, out / "historical_site_year_fusion_summary.json")
write_json(out / "manifest.json", manifest)
display(base.head(20))